# 1 · Agente de suporte que lembra do atendimento
**Café IA · Floripa | LangGraph + Oracle Agent Memory | 20 minutos**

Marina está com um problema de VPN. Vamos criar um agente, conversar com ele e observar
o que fica na memória Oracle: histórico, resumo da conversa e memórias duradouras. Depois começamos
outro atendimento e recuperamos o contexto sem enviar a conversa anterior ao agente.

Roteiro: **criar agente → conversar → mostrar resumo de short-term memory → buscar long-term
memory → nova conversa → definir campos e metadados**. Antes, fazemos uma comparação sem memória.

O LangGraph orquestra o agente; o SDK Oracle persiste e extrai a memória [referência](https://docs.oracle.com/en/database/oracle/agent-memory/26.4/guide/int-langgraph.html)

**Preparação:** Configure o `.env` local e execute de cima para baixo.

In [6]:
%%capture
!pip install pandas litellm langchain_core langgraph oracleagentmemory

In [7]:
from uuid import uuid4
import pandas as pd
from IPython.display import display
from litellm import acompletion
from langchain_core.messages import AIMessage, HumanMessage
from langgraph.graph import START, END, MessagesState, StateGraph
from oracleagentmemory.core import (
    OracleAgentMemory, MemoryExtractionConfig, SchemaPolicy, SearchStrategy,
)
from oracleagentmemory.core.extractors.memoryextractionconfig import MemoryExtractionMode
from oracleagentmemory.core.llms.llm import Llm
from apoio import conectar, parametros_llm

pd.set_option('display.max_colwidth', 120)
conexao = conectar()
print('Autonomous conectado; Oracle', conexao.version)
usuario = 'marina-' + uuid4().hex[:8]
agente_id = 'suporte-floripa'
ticket = 'SUP-1042'

Autonomous conectado; Oracle 23.26.4.1.0


## 1 · Configurar a memória do agente

| Camada | O que guarda | Escopo |
|---|---|---|
| Short-term memory | Mensagens e resumo da conversa | Thread do atendimento |
| Long-term memory | Nome, chamado, fatos confirmados e preferências | Usuário/agente, recuperáveis em outra thread |
| Metadados | Ticket, canal, categoria, origem e aprovação | Rótulos para filtrar os registros |

O SDK usa um LLM para extrair memórias e resumir o histórico. `INLINE` faz a extração terminar
antes de devolver o controle à célula, facilitando a apresentação. Os valores `-1` em
frequência/janela extraem uma vez por lote, incluindo a pergunta e a resposta completas.
A cada par de mensagens, o resumo pode ser atualizado. A política orienta quais fatos
devem virar memórias.

Usamos `KEYWORD` (busca textual no Oracle) para não exigir um endpoint de embeddings.
O store `CAFE_SUPORTE` é exclusivo deste lab. Para uma solução mais robusta, `VECTOR` e `HYBRID` são extensões possíveis,
mas exigem a configuração de embeddings e do store.

In [ ]:
politica = MemoryExtractionConfig(
    extract_memories=True,  # Extrai memórias duradouras automaticamente
    extraction_mode=MemoryExtractionMode.INLINE,  # Aguarda a extração terminar
    memory_extraction_frequency=-1,  # Uma extração por lote de mensagens
    memory_extraction_window=-1,  # Considera todo o lote recém-adicionado
    enable_context_summary=True,  # Mantém um resumo da conversa
    context_summary_update_frequency=2,  # Limiar de 2 novas mensagens para atualizar
    memory_extraction_inherit_message_metadata=['ticket', 'canal'],  # Herda estes rótulos
    memory_extraction_custom_instructions=(  # Orientações sobre quais fatos guardar
        'Extraia somente fatos confirmados pelo usuário: nome, ticket de suporte, '
        'sistema afetado, erro relatado, tentativas já feitas e preferências de atendimento. '
        'Preserve o contexto do suporte de VPN nas memórias. '
        'Não grave senhas, tokens, códigos temporários ou suposições do assistente. '
        'Diferencie o estado deste ticket de uma preferência estável do cliente.'
    ),
)
llm_memoria = Llm(**parametros_llm(), temperature=0)  
memoria = OracleAgentMemory(
    connection=conexao,  
    llm=llm_memoria, 
    memory_store_id='CAFE_SUPORTE', 
    schema_policy=SchemaPolicy.CREATE_IF_NECESSARY,  # Cria objetos do SDK se faltarem
    search_strategy=SearchStrategy.KEYWORD,  # Busca textual, sem embeddings
    memory_extraction_config=politica,  
)
thread = await memoria.create_thread_async(
    user_id=usuario, 
    agent_id=agente_id,  
    metadata={'ticket': ticket, 'canal': 'chat'},  
)
print('Usuário fictício:', usuario)
print('Thread do atendimento:', thread.thread_id)

Usuário fictício: marina-82f2fc6e
Thread do atendimento: f61a9533-7b72-47a4-a334-f062697da479


## 2 · Criar o agente com LangGraph

O fluxo tem três nós: **recuperar contexto → responder → persistir o turno**.
O modo sem memória usa o mesmo modelo, mas não lê nem grava contexto.
No primeiro atendimento, usamos o resumo da thread. A busca de longo prazo será ligada
depois, quando começarmos uma conversa nova.

Cada chamada ao grafo recebe somente a pergunta atual. Não usamos um checkpointer local:
a continuidade entre chamadas virá dos dados do Oracle. A busca é explícita no fluxo
customizado da referência; o modelo não precisa decidir quando chamar uma ferramenta.

In [ ]:
class EstadoAtendimento(MessagesState):
    thread_id: str  # Identificador da conversa no Oracle.
    usar_memoria: bool  # Liga a leitura e a gravação de memória.
    consultar_ltm: bool  # Inclui a busca de memórias duradouras.
    resumo: str  # Resumo de curto prazo recuperado da thread.
    lembrancas: list[str]  # Fatos e preferências recuperados da memória longa.

async def buscar_longo_prazo():
    # KEYWORD: usamos termos explícitos para os assuntos desta demonstração.
    encontrados = {}
    for consulta in ['VPN', 'Marina', ticket, 'português', 'email']:
        resultados = await memoria.search_async(
            consulta,  # Termo pesquisado na memória.
            user_id=usuario,  # Restringe ao cliente atual.
            agent_id=agente_id,  # Restringe ao agente atual.
            exact_user_match=True,  # Exige correspondência exata do cliente.
            exact_agent_match=True,  # Exige correspondência exata do agente.
            record_types=[
                'fact',  # Fato confirmado.
                'preference',  # Preferência do cliente.
                'memory',  # Lembrança geral.
                'guideline',  # Orientação duradoura.
            ],
            metadata_filter={'ticket': ticket},  # Filtra pelo chamado.
            max_results=8,  # Até 8 resultados por termo pesquisado.
        )
        for resultado in resultados:
            encontrados[resultado.id] = resultado
    return list(encontrados.values())

async def recuperar_contexto(estado):
    if not estado['usar_memoria']:
        return {'resumo': '', 'lembrancas': []}
    conversa = await memoria.get_thread_async(estado['thread_id'])
    mensagens = await conversa.get_messages_async()
    resumo = (await conversa.get_summary_async()).content if mensagens else ''
    resultados = await buscar_longo_prazo() if estado['consultar_ltm'] else []
    return {'resumo': resumo, 'lembrancas': [r.content for r in resultados]}

async def responder(estado):
    instrucao = (
        'Você é um agente de suporte e atendimento. Responda em português em até três frases. '
        'Use apenas os fatos do contexto e da pergunta. Se não souber o nome, ticket ou '
        'preferências, diga que não sabe. Não invente diagnósticos ou resolução. '
        'Não solicite segredos, não execute comandos e não afirme abrir tickets externos.\n'
        'Resumo da conversa atual: ' + estado['resumo'] + '\n'
        'Memórias recuperadas de outros atendimentos:\n' + '\n'.join(estado['lembrancas'])
    )
    retorno = await acompletion(
        **parametros_llm(),
        messages=[{'role': 'system', 'content': instrucao},
                  {'role': 'user', 'content': estado['messages'][-1].content}],
        temperature=0, max_tokens=400,
    )
    return {'messages': [AIMessage(content=retorno.choices[0].message.content)]}

async def persistir_turno(estado):
    if estado['usar_memoria']:
        conversa = await memoria.get_thread_async(estado['thread_id'])
        await conversa.add_messages_async(
            [{'role': 'user', 'content': estado['messages'][-2].content},
             {'role': 'assistant', 'content': estado['messages'][-1].content}],
            metadata={'ticket': ticket, 'canal': 'chat'},  # Rótulos salvos com o turno.
        )
    return {}

fluxo = StateGraph(EstadoAtendimento)
fluxo.add_node('recuperar_contexto', recuperar_contexto)
fluxo.add_node('responder', responder)
fluxo.add_node('persistir_turno', persistir_turno)
fluxo.add_edge(START, 'recuperar_contexto')
fluxo.add_edge('recuperar_contexto', 'responder')
fluxo.add_edge('responder', 'persistir_turno')
fluxo.add_edge('persistir_turno', END)
agente = fluxo.compile()

async def conversar(texto, *, com_memoria=True, longo_prazo=False, thread_id=None):
    saida = await agente.ainvoke({
        'messages': [HumanMessage(content=texto)],
        'thread_id': thread_id or thread.thread_id,
        'usar_memoria': com_memoria,
        'consultar_ltm': longo_prazo,
    })
    print('Cliente:', texto)
    print('Agente:', saida['messages'][-1].content, '\n')
    return saida

## 3 · Comparação: sem memória, a segunda chamada não lembra
Primeiro apresentamos nome e ticket. Depois perguntamos pelos mesmos dados em outra
invocação. A variável Python da primeira resposta não é enviada na segunda chamada.
O agente deve reconhecer que não recebeu essas informações.

In [4]:
sem_memoria_1 = await conversar(
    'Sou Marina Costa e meu chamado de VPN é SUP-1042.', com_memoria=False,
)
sem_memoria_2 = await conversar(
    'Qual meu nome e o número do meu chamado?', com_memoria=False,
)

Cliente: Sou Marina Costa e meu chamado de VPN é SUP-1042.
Agente: Olá, Marina Costa. Entendi que seu chamado de VPN é o SUP-1042. 



Cliente: Qual meu nome e o número do meu chamado?
Agente: Não tenho acesso ao seu nome ou ao número do seu chamado com as informações disponíveis. 



## 4 · Conversar com o agente e deixar o SDK extrair a memória
Agora cada turno é salvo no Oracle. O histórico e o resumo continuam na mesma thread;
o SDK também extrai fatos e preferências para a memória de longo prazo.

Observe a diferença entre **o problema do ticket** e **como Marina prefere ser atendida**.
As respostas são produzidas pelo modelo configurado; não são um diálogo pré-gravado.

In [5]:
falas = [
    'Sou Marina Costa. Meu ticket é SUP-1042: a VPN desconecta no Windows 11. '
    'No suporte da VPN, prefiro respostas curtas em português e contato por email.',
    'Já reiniciei o notebook e a VPN, mas o erro VPN-809 continua. '
    'Por favor, não sugira novamente reiniciar como se eu ainda não tivesse tentado.',
    'Ainda não resolvemos. O próximo passo combinado é coletar logs de diagnóstico '
    'sem senhas e enviar ao suporte por email. O ticket continua em análise.',
]
for fala in falas:
    turno = await conversar(fala)

Cliente: Sou Marina Costa. Meu ticket é SUP-1042: a VPN desconecta no Windows 11. No suporte da VPN, prefiro respostas curtas em português e contato por email.
Agente: Olá, Marina Costa. Entendi que seu ticket é SUP-1042 e que a VPN está desconectando no Windows 11. Compreendi que prefere respostas curtas em português e contato por e-mail. 



Cliente: Já reiniciei o notebook e a VPN, mas o erro VPN-809 continua. Por favor, não sugira novamente reiniciar como se eu ainda não tivesse tentado.
Agente: Entendido, Marina. Compreendo que você já reiniciou o notebook e a VPN, e o erro VPN-809 persiste. Como agente de suporte, não tenho informações para diagnosticar ou resolver esse problema diretamente. 



Cliente: Ainda não resolvemos. O próximo passo combinado é coletar logs de diagnóstico sem senhas e enviar ao suporte por email. O ticket continua em análise.
Agente: Entendido, Marina. O próximo passo é coletar os logs de diagnóstico sem senhas e enviá-los por e-mail, conforme combinado. O ticket SUP-1042 continua em análise. 



## 5 · Mostrar o resumo de short-term memory
O resumo é gerado pelo SDK com o LLM e fica associado à thread. Ele condensa o contexto
atual: problema, tentativas, preferências relevantes e próximo passo. Não é uma string fixa.
A tabela abaixo mostra também as mensagens originais, para comparar histórico e resumo.

In [6]:
resumo = await thread.get_summary_async()
print('RESUMO DO ATENDIMENTO:\n', resumo.content)
assert resumo.content.strip(), 'O resumo deveria ter conteúdo.'
display(pd.DataFrame([
    {'papel': m.role, 'conteudo': m.content} for m in await thread.get_messages_async()
]))

RESUMO DO ATENDIMENTO:
 Marina Costa (ticket SUP-1042) reports her VPN disconnects on Windows 11 with error VPN-809. She has already tried restarting her notebook and VPN and requests not to be asked to do so again. She prefers short responses in Portuguese and email contact for support. The support agent acknowledges her actions but states they cannot directly diagnose or resolve the issue. The next step is for Marina to collect diagnostic logs (without passwords) and send them to support via email. The ticket remains under analysis.


,papel,conteudo
0,user,"Sou Marina Costa. Meu ticket é SUP-1042: a VPN desconecta no Windows 11. No suporte da VPN, prefiro respostas curtas..."
1,assistant,"Olá, Marina Costa. Entendi que seu ticket é SUP-1042 e que a VPN está desconectando no Windows 11. Compreendi que pr..."
2,user,"Já reiniciei o notebook e a VPN, mas o erro VPN-809 continua. Por favor, não sugira novamente reiniciar como se eu a..."
3,assistant,"Entendido, Marina. Compreendo que você já reiniciou o notebook e a VPN, e o erro VPN-809 persiste. Como agente de su..."
4,user,Ainda não resolvemos. O próximo passo combinado é coletar logs de diagnóstico sem senhas e enviar ao suporte por ema...
5,assistant,"Entendido, Marina. O próximo passo é coletar os logs de diagnóstico sem senhas e enviá-los por e-mail, conforme comb..."


## 6 · Buscar informações dentro da long-term memory
Esta busca retorna registros extraídos pelo SDK, não a transcrição inteira.
Mantemos `user_id` e `agent_id` para isolar o cliente e filtramos pelo ticket nos metadados.
Tipos e textos extraídos podem variar conforme o modelo; compare-os com o resumo anterior.
Como a busca é textual, usamos termos explícitos sobre VPN, cliente, ticket e preferências,
reunindo os resultados sem duplicatas. Em Jupyter usamos as APIs assíncronas com `await`.

In [7]:
memorias = await buscar_longo_prazo()
assert memorias, 'Esperávamos memórias extraídas da conversa sobre VPN.'
display(pd.DataFrame([
    {'tipo': r.record.record_type, 'conteudo': r.content,
     'metadados': r.record.metadata} for r in memorias
]))

,tipo,conteudo,metadados
0,fact,A VPN está desconectando no Windows 11.,"{'ticket': 'SUP-1042', 'canal': 'chat', '$agent_memory': {'extraction_id': 'vpn_issue_windows_11', 'scope': 'environ..."
1,fact,A usuária está enfrentando o erro VPN-809.,"{'ticket': 'SUP-1042', 'canal': 'chat', '$agent_memory': {'extraction_id': 'vpn_error_vpn_809', 'scope': 'user', 'im..."
2,fact,A usuária já reiniciou a VPN como tentativa de solução para o problema.,"{'ticket': 'SUP-1042', 'canal': 'chat', '$agent_memory': {'extraction_id': 'troubleshooting_restarted_vpn', 'scope':..."
3,fact,O ticket de suporte SUP-1042 para o problema de VPN ainda não foi resolvido.,"{'ticket': 'SUP-1042', 'canal': 'chat', '$agent_memory': {'extraction_id': 'vpn_ticket_unresolved_sup1042', 'scope':..."
4,fact,O ticket de suporte SUP-1042 para o problema de VPN continua em análise.,"{'ticket': 'SUP-1042', 'canal': 'chat', '$agent_memory': {'extraction_id': 'vpn_ticket_under_analysis_sup1042', 'sco..."
5,fact,A usuária já reiniciou o notebook como tentativa de solução para o problema da VPN.,"{'ticket': 'SUP-1042', 'canal': 'chat', '$agent_memory': {'extraction_id': 'troubleshooting_restarted_notebook', 'sc..."
6,guideline,"Não sugerir à usuária reiniciar o notebook ou a VPN novamente, pois ela já tentou essas ações.","{'ticket': 'SUP-1042', 'canal': 'chat', '$agent_memory': {'extraction_id': 'guideline_no_suggest_restart_vpn_noteboo..."
7,preference,A usuária prefere contato por e-mail para o suporte da VPN.,"{'ticket': 'SUP-1042', 'canal': 'chat', '$agent_memory': {'extraction_id': 'communication_preference_contact_email',..."
8,fact,O nome da usuária é Marina Costa.,"{'ticket': 'SUP-1042', 'canal': 'chat', '$agent_memory': {'extraction_id': 'user_name_marina_costa', 'scope': 'user'..."
9,fact,O nome da usuária é Marina Costa.,"{'ticket': 'SUP-1042', 'canal': 'chat', '$agent_memory': {'extraction_id': 'user_name_marina', 'scope': 'user', 'imp..."


## 7 · Uma nova conversa recupera as lembranças do cliente
A nova thread começa sem mensagens e sem resumo. Não passamos `falas`, `turno` ou o resumo
anterior ao grafo. A busca no Oracle injeta as memórias duradouras na resposta.
Uma nova thread não significa um novo usuário: usamos as mesmas identidades do cliente e agente.

In [8]:
nova_thread = await memoria.create_thread_async(
    user_id=usuario,  # Mesmo cliente, em uma conversa nova.
    agent_id=agente_id,  # Mantém a identidade do agente.
    metadata={'ticket': ticket, 'canal': 'chat'},  # Associa a nova conversa ao chamado.
)
assert await nova_thread.get_messages_async() == []
nova_sessao = await conversar(
    'Estou voltando ao suporte da VPN. Qual é meu chamado, o que já tentei '
    'e como prefiro receber as orientações?',
    longo_prazo=True, thread_id=nova_thread.thread_id,
)
print('Resumo inicial da nova thread recebido pelo grafo:', repr(nova_sessao['resumo']))
print('Memórias de longo prazo recuperadas:', len(nova_sessao['lembrancas']))
assert nova_sessao['resumo'] == '' and nova_sessao['lembrancas']

Cliente: Estou voltando ao suporte da VPN. Qual é meu chamado, o que já tentei e como prefiro receber as orientações?
Agente: Seu chamado é o SUP-1042. Você já tentou reiniciar a VPN e o notebook. Você prefere receber as orientações por e-mail e em respostas curtas. 

Resumo inicial da nova thread recebido pelo grafo: ''
Memórias de longo prazo recuperadas: 14


## 8 · Consultar diretamente o que ficou no banco
O prefixo vem do `memory_store_id='CAFE_SUPORTE'`. As consultas usam somente as tabelas
deste lab e filtram a thread original. Compare: mensagem bruta, resumo da thread e fatos
duradouros são registros diferentes. A coluna `METADATA` contém os rótulos herdados.

In [9]:
from oracledb import LOB

def consultar(sql, **parametros):
    with conexao.cursor() as cursor:
        cursor.execute(sql, parametros)
        nomes = [d[0] for d in cursor.description]
        linhas = [[v.read() if isinstance(v, LOB) else v for v in linha] for linha in cursor]
    return pd.DataFrame(linhas, columns=nomes)

print('LONG-TERM MEMORY')
display(consultar('''
    SELECT MEMORY_TYPE, CONTENT, METADATA FROM CAFE_SUPORTE_MEMORY
    WHERE THREAD_ID=:id ORDER BY ORDER_SEQ
''', id=thread.thread_id))
print('HISTÓRICO / SHORT-TERM MEMORY')
display(consultar('''
    SELECT MESSAGE_ROLE, CONTENT, METADATA FROM CAFE_SUPORTE_MESSAGE
    WHERE THREAD_ID=:id ORDER BY ORDER_SEQ
''', id=thread.thread_id))
print('RESUMO PERSISTIDO')
display(consultar('''
    SELECT CONTENT FROM CAFE_SUPORTE_THREAD_SUMMARY WHERE THREAD_ID=:id
''', id=thread.thread_id))

LONG-TERM MEMORY


,MEMORY_TYPE,CONTENT,METADATA
0,preference,A usuária prefere respostas curtas em português para o suporte da VPN.,"{'ticket': 'SUP-1042', 'canal': 'chat', '$agent_memory': {'extraction_id': 'communication_preference_short_portugues..."
1,preference,A usuária prefere contato por e-mail para o suporte da VPN.,"{'ticket': 'SUP-1042', 'canal': 'chat', '$agent_memory': {'extraction_id': 'communication_preference_contact_email',..."
2,fact,O nome da usuária é Marina Costa.,"{'ticket': 'SUP-1042', 'canal': 'chat', '$agent_memory': {'extraction_id': 'user_name_marina_costa', 'scope': 'user'..."
3,fact,O número do ticket de suporte é SUP-1042.,"{'ticket': 'SUP-1042', 'canal': 'chat', '$agent_memory': {'extraction_id': 'support_ticket_sup_1042', 'scope': 'envi..."
4,fact,A VPN está desconectando no Windows 11.,"{'ticket': 'SUP-1042', 'canal': 'chat', '$agent_memory': {'extraction_id': 'vpn_issue_windows_11', 'scope': 'environ..."
5,guideline,"Não sugerir à usuária reiniciar o notebook ou a VPN novamente, pois ela já tentou essas ações.","{'ticket': 'SUP-1042', 'canal': 'chat', '$agent_memory': {'extraction_id': 'guideline_no_suggest_restart_vpn_noteboo..."
6,fact,O nome da usuária é Marina Costa.,"{'ticket': 'SUP-1042', 'canal': 'chat', '$agent_memory': {'extraction_id': 'user_name_marina', 'scope': 'user', 'imp..."
7,fact,A usuária está enfrentando o erro VPN-809.,"{'ticket': 'SUP-1042', 'canal': 'chat', '$agent_memory': {'extraction_id': 'vpn_error_vpn_809', 'scope': 'user', 'im..."
8,fact,A usuária já reiniciou o notebook como tentativa de solução para o problema da VPN.,"{'ticket': 'SUP-1042', 'canal': 'chat', '$agent_memory': {'extraction_id': 'troubleshooting_restarted_notebook', 'sc..."
9,fact,A usuária já reiniciou a VPN como tentativa de solução para o problema.,"{'ticket': 'SUP-1042', 'canal': 'chat', '$agent_memory': {'extraction_id': 'troubleshooting_restarted_vpn', 'scope':..."


HISTÓRICO / SHORT-TERM MEMORY


,MESSAGE_ROLE,CONTENT,METADATA
0,user,"[{'type': 'text', 'text': 'Sou Marina Costa. Meu ticket é SUP-1042: a VPN desconecta no Windows 11. No suporte da VP...","{'ticket': 'SUP-1042', 'canal': 'chat'}"
1,assistant,"[{'type': 'text', 'text': 'Olá, Marina Costa. Entendi que seu ticket é SUP-1042 e que a VPN está desconectando no Wi...","{'ticket': 'SUP-1042', 'canal': 'chat'}"
2,user,"[{'type': 'text', 'text': 'Já reiniciei o notebook e a VPN, mas o erro VPN-809 continua. Por favor, não sugira novam...","{'ticket': 'SUP-1042', 'canal': 'chat'}"
3,assistant,"[{'type': 'text', 'text': 'Entendido, Marina. Compreendo que você já reiniciou o notebook e a VPN, e o erro VPN-809 ...","{'ticket': 'SUP-1042', 'canal': 'chat'}"
4,user,"[{'type': 'text', 'text': 'Ainda não resolvemos. O próximo passo combinado é coletar logs de diagnóstico sem senhas ...","{'ticket': 'SUP-1042', 'canal': 'chat'}"
5,assistant,"[{'type': 'text', 'text': 'Entendido, Marina. O próximo passo é coletar os logs de diagnóstico sem senhas e enviá-lo...","{'ticket': 'SUP-1042', 'canal': 'chat'}"


RESUMO PERSISTIDO


,CONTENT
0,{'text': 'Marina Costa (ticket SUP-1042) reports her VPN disconnects on Windows 11. She prefers short responses in P...
1,{'text': 'Marina Costa (ticket SUP-1042) reports her VPN disconnects on Windows 11 with error VPN-809. She has alrea...
2,{'text': 'Marina Costa (ticket SUP-1042) reports her VPN disconnects on Windows 11 with error VPN-809. She has alrea...


## 9 · Definir exatamente quais campos e metadados salvar
Até aqui o LLM escolheu os fatos conforme as instruções de extração. Agora mostramos uma
política determinística para dados estruturados de atendimento: uma lista permitida e
consentimento explícito. Ela complementa a extração automática; não a substitui retroativamente.

**Conteúdo** é o fato que o agente recupera. **Metadados** são os rótulos associados ao fato:
nome, ticket, categoria, origem e aprovação. A preferência também fica no conteúdo para ser
pesquisável. Escolher metadados não elimina dados que já foram enviados no histórico.

A lista permitida protege este caminho estruturado. Instruções de extração ao LLM são
orientações, não uma garantia de remoção de segredos. Não envie credenciais reais ao agente.

In [10]:
cadastro = {
    'nome': 'Marina Costa',
    'ticket': ticket,
    'preferencias': 'atendimento em português, com respostas curtas e contato por email',
    'senha': '<CAMPO FICTÍCIO QUE NÃO DEVE SER SALVO>',
}
campos_permitidos = {'nome', 'ticket', 'preferencias'}  # Campos aceitos neste cadastro.
consentimento = True  # Autoriza a gravação dos campos permitidos.
decisoes = []
for campo, valor in cadastro.items():
    salvar = consentimento and campo in campos_permitidos
    decisoes.append({'campo': campo, 'salvar': salvar,
                     'motivo': 'permitido e consentido' if salvar else 'campo bloqueado'})
    if salvar:
        await memoria.add_memory_async(
            f'Cadastro de atendimento VPN: {campo} = {valor}.',
            memory_type='preference' if campo == 'preferencias' else 'fact',  # Tipo da memória.
            user_id=usuario,  # Cliente dono do registro.
            agent_id=agente_id,  # Agente associado ao registro.
            thread_id=thread.thread_id,  # Conversa de origem.
            metadata={  # Rótulos usados nos filtros de busca.
                'nome': cadastro['nome'],  # Nome associado ao cadastro.
                'ticket': ticket,  # Chamado relacionado.
                'categoria': campo,  # Campo representado por esta memória.
                'origem': 'cadastro_consentido',  # Fonte do dado.
                'aprovado': True,  # Marca o cadastro como aprovado.
            },
            autonomous_linking=False,  # Não pede ao LLM para criar ligações entre memórias.
        )
display(pd.DataFrame(decisoes))

,campo,salvar,motivo
0,nome,True,permitido e consentido
1,ticket,True,permitido e consentido
2,preferencias,True,permitido e consentido
3,senha,False,campo bloqueado


## 10 · Recuperar somente o que foi aprovado para esse ticket
O filtro não é texto adicionado ao prompt: ele é aplicado pelo SDK na consulta ao Oracle.
Usamos o escopo do cliente e agente **junto** dos metadados; rótulos de ticket não substituem
o isolamento por identidade. `categoria` permite buscar só as preferências.

In [11]:
preferencias = await memoria.search_async(
    'atendimento',  # Termo pesquisado no conteúdo.
    user_id=usuario,  # Cliente atual.
    agent_id=agente_id,  # Agente atual.
    exact_user_match=True,  # Exige o mesmo cliente.
    exact_agent_match=True,  # Exige o mesmo agente.
    record_types=['preference'],  # Retorna apenas preferências.
    metadata_filter={  # Todos estes critérios precisam corresponder.
        'ticket': ticket,  # Mesmo chamado.
        'categoria': 'preferencias',  # Apenas o campo de preferências.
        'origem': 'cadastro_consentido',  # Apenas o cadastro explícito.
        'aprovado': True,  # Apenas registros marcados como aprovados.
    },
    max_results=5,  # Limita a busca a 5 resultados.
)
assert preferencias, 'O cadastro aprovado deveria ser recuperado.'
display(pd.DataFrame([
    {'conteudo': r.content, 'metadados': r.record.metadata} for r in preferencias
]))
outra_origem = await memoria.search_async(
    'atendimento',  # Mantém o termo da busca anterior.
    user_id=usuario, agent_id=agente_id,  # Mantém o escopo do cliente e agente.
    exact_user_match=True, exact_agent_match=True,  # Correspondência exata do escopo.
    record_types=['preference'],  # Apenas preferências.
    metadata_filter={'ticket': 'SUP-INEXISTENTE', 'origem': 'cadastro_consentido'},  # Outro ticket.
    max_results=5,  # Até 5 resultados; esperamos nenhum neste caso.
)
assert not outra_origem
print('Mesmo termo com outro ticket: nenhum registro retornado.')

,conteudo,metadados
0,"Cadastro de atendimento VPN: preferencias = atendimento em português, com respostas curtas e contato por email.","{'nome': 'Marina Costa', 'ticket': 'SUP-1042', 'categoria': 'preferencias', 'origem': 'cadastro_consentido', 'aprova..."


Mesmo termo com outro ticket: nenhum registro retornado.


## 11 · Conferir no SQL e reabrir por outra conexão
Verificamos que a memória estruturada contém os metadados escolhidos e que o campo
`senha` não foi gravado por esse caminho. Por fim, outra conexão ao Oracle reabre o resumo.
A persistência não depende do estado Python do LangGraph.

In [12]:
registros_cadastro = consultar('''
    SELECT CONTENT, METADATA FROM CAFE_SUPORTE_MEMORY
    WHERE THREAD_ID=:id
      AND JSON_VALUE(METADATA, '$.origem')='cadastro_consentido'
    ORDER BY ORDER_SEQ
''', id=thread.thread_id)
display(registros_cadastro)
assert len(registros_cadastro) == 3
assert not any('CAMPO FICTÍCIO' in str(c) for c in registros_cadastro.CONTENT)

outra_conexao = conectar()
memoria_reaberta = OracleAgentMemory(
    connection=outra_conexao,  # Usa uma conexão independente.
    llm=llm_memoria,  # Reutiliza o modelo de extração/resumo.
    memory_store_id='CAFE_SUPORTE',  # Reabre o mesmo store persistido.
    schema_policy=SchemaPolicy.REQUIRE_EXISTING,  # Exige objetos já criados; sem DDL.
    search_strategy=SearchStrategy.KEYWORD,  # Mantém a busca textual.
    memory_extraction_config=politica,  # Mantém a mesma política de memória.
)
recuperada = await memoria_reaberta.get_thread_async(thread.thread_id)
resumo_reaberto = (await recuperada.get_summary_async()).content
print('RESUMO REABERTO EM OUTRA CONEXÃO:\n', resumo_reaberto)
assert resumo_reaberto == resumo.content
await memoria_reaberta.close_async()
outra_conexao.close()
await memoria.close_async()
conexao.close()
print('Conexões encerradas; dados fictícios preservados para inspeção.')

,CONTENT,METADATA
0,Cadastro de atendimento VPN: nome = Marina Costa.,"{'nome': 'Marina Costa', 'ticket': 'SUP-1042', 'categoria': 'nome', 'origem': 'cadastro_consentido', 'aprovado': True}"
1,Cadastro de atendimento VPN: ticket = SUP-1042.,"{'nome': 'Marina Costa', 'ticket': 'SUP-1042', 'categoria': 'ticket', 'origem': 'cadastro_consentido', 'aprovado': T..."
2,"Cadastro de atendimento VPN: preferencias = atendimento em português, com respostas curtas e contato por email.","{'nome': 'Marina Costa', 'ticket': 'SUP-1042', 'categoria': 'preferencias', 'origem': 'cadastro_consentido', 'aprova..."


RESUMO REABERTO EM OUTRA CONEXÃO:
 Marina Costa (ticket SUP-1042) reports her VPN disconnects on Windows 11 with error VPN-809. She has already tried restarting her notebook and VPN and requests not to be asked to do so again. She prefers short responses in Portuguese and email contact for support. The support agent acknowledges her actions but states they cannot directly diagnose or resolve the issue. The next step is for Marina to collect diagnostic logs (without passwords) and send them to support via email. The ticket remains under analysis.
Conexões encerradas; dados fictícios preservados para inspeção.
